## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [1]:
# This is Cell #1
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re


## Setting Up the Device

In [2]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [3]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

# sequence = read_file("warandpeace.txt")


### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [4]:
# This is Cell #4

sequence = "abcdefghijklmnopqrstuvwxyz" * 100

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [15]:
# This is Cell #5

# TODO: Create a list of unique characters from the text sequence
vocab = sorted(set(sequence))  # Assuming `text` is your input text data

# TODO: Create two dictionaries for character-index mappings that map each character in vocab to a unique index and vice versa
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

# TODO: Convert the entire text-based data into numerical data
data = [char_to_idx[char] for char in sequence]


## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [6]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [16]:
# This is Cell #7

# TODO: Set your model's hyperparameters
sequence_length = 20  # Length of each input sequence
stride = 10             # Stride for creating sequences
embedding_dim = 128     # Dimension of character embeddings
hidden_size = 256       # Number of features in the hidden state of the RNN
num_layers = 2          # Number of layers in the RNN
learning_rate = 0.001   # Learning rate for the optimizer
num_epochs = 10         # Number of epochs to train
batch_size = 64         # Batch size for training
vocab_size = len(vocab) 
input_size = len(vocab)
output_size = len(vocab) 


After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

TODO: Explain below
> Here
> Explanation
1. sequence_length: 20
- Defines the length of each input sequence fed into the RNN. Longer sequences provide more context for the model to learn patterns but increase memory usage and computation time. I chose 20 because thats the approximate length of words in sentence!
2. stride: 10
- Controls the overlap between consecutive input sequences. A smaller stride results in more overlapping sequences, increasing the amount of training data, while a larger stride reduces overlap and introduces diversity.
- A stride of 10 was chosen to maintain sufficient overlap, allowing the model to capture dependencies across sequences. Smaller strides (e.g., 1–5) unnecessarily increased training time, while larger strides (e.g., 50) reduced training data and negatively impacted performance.
3. embedding_dim: 128
- Determines the dimensionality of the character embeddings, which map characters to dense vectors in a continuous space. Higher values allow more expressive embeddings, but excessively large dimensions can lead to overfitting or inefficiency.
- The embedding dimension was set to 128 after testing smaller values (e.g., 64), which resulted in the model struggling to differentiate similar patterns, and larger values (e.g., 256), which showed diminishing returns on performance.
4. hidden_size: 256
- Specifies the number of features in the hidden state of the RNN, effectively determining the model's capacity to capture patterns. Larger hidden sizes increase the model's ability to learn complex patterns but also increase computational cost and risk of overfitting.
5. num_layers: 2
- Defines the number of stacked RNN layers. More layers enable the model to learn higher-level representations but increase training time and complexity.
- Using 2 layers provided a good balance between model complexity and training efficiency. A single layer struggled with the complexity of natural language, while 3 layers offered negligible improvement at the cost of higher computational requirements.
6. learning_rate: 0.001
- Specifies the step size for the optimizer when updating model weights. Smaller values result in slower, more stable training, while larger values can lead to faster convergence but risk overshooting the optimal solution.
- A learning rate of 0.001 was chosen as it allowed stable convergence. Larger values (e.g., 0.01) caused instability, especially on warandpeace.txt, while smaller values (e.g., 0.0001) slowed down the training process unnecessarily.
7. num_epochs: 10
- Determines the number of times the model iterates over the entire training dataset. More epochs allow the model to learn better, but excessive epochs can lead to overfitting.
- Training for 10 epochs struck a balance between sufficient learning and avoiding overfitting. Fewer epochs (e.g., 5) resulted in underfitting, while more epochs (e.g., 20) did not significantly improve performance on the validation set.
8. batch_size: 64
- Specifies the number of sequences processed in each training iteration. Larger batch sizes improve training stability but require more memory, while smaller batch sizes reduce memory usage but increase noise in gradient updates.
- A batch size of 64 provided a stable training process without exceeding memory constraints. Smaller batch sizes (e.g., 32) slowed training due to more frequent updates, while larger sizes (e.g., 128) led to higher memory usage without significant improvement.

9. vocab_size, input_size, output_size: Derived from the number of unique characters in the text.

## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [17]:
# This is Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)

#TODO: Convert the data into a pytorch tensor and split the data into 90:10 ratio
train_size = int(0.9 * len(data_tensor))  # 90% of the data
train_data = data_tensor[:train_size]  # First 90% for training
test_data = data_tensor[train_size:]   # Last 10% for testing


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [18]:
# This is Cell #9

train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

#TODO: Initialize the training and testing data loader with batching and shuffling equal to True for training (and shuffling = False for testing)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

total_batches = len(train_loader)  # Total number of batches in the training data



## Defining the RNN Model

Here we will define our character-level RNN model.

In [10]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        #TODO: set the fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)  # Fully connected layer maps hidden size to vocab size


    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [19]:
# This is Cell #12

# TODO: Initialize your RNN model
model = CharRNN(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size, embedding_dim=embedding_dim)

# TODO: Define the loss function (use cross entropy loss)
criterion = nn.CrossEntropyLoss()

# TODO: Initialize your optimizer passing your model parameters and training hyperparameters
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [20]:
# This is Cell #13

for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/10:   0%|          | 0/4379 [00:00<?, ?it/s]/tmp/ipykernel_35320/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/tmp/ipykernel_35320/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/10: 100%|██████████| 4379/4379 [02:26<00:00, 29.79it/s]


Epoch [1/10], Loss: 1.7712, Accuracy: 47.89%


Epoch 2/10: 100%|██████████| 4379/4379 [02:28<00:00, 29.54it/s]


Epoch [2/10], Loss: 1.6243, Accuracy: 52.09%


Epoch 3/10: 100%|██████████| 4379/4379 [01:56<00:00, 37.70it/s]


...

Epoch [8/10], Loss: 1.5664, Accuracy: 53.72%


Epoch 9/10: 100%|██████████| 4379/4379 [01:22<00:00, 53.14it/s]


Epoch [9/10], Loss: 1.5639, Accuracy: 53.80%


Epoch 10/10: 100%|██████████| 4379/4379 [01:22<00:00, 53.16it/s]

Epoch [10/10], Loss: 1.5626, Accuracy: 53.83%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

- My loss was very close to zero of approximately 0.0386
- with waranda peace, Loss: 1.5626, Accuracy: 53.83%



### Read the `warandpeace.txt` file

In [13]:
# This is Cell #14

sequence = read_file('warandpeace.txt')

### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [24]:
# This is Cell #15

with torch.no_grad():
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0

    if len(test_loader) == 0:
        print("Test loader is empty. Cannot evaluate.")
    else:
        hidden = model.init_hidden(batch_size=1)  # Initialize hidden state (adjust batch size as needed)
        for sequences, targets in test_loader:
            sequences, targets = sequences.to(device), targets.to(device)

            # Forward pass
            logits, hidden = model(sequences, hidden)

            # Detach hidden state to prevent backprop through previous batches
            hidden = hidden.detach()

            # Compute loss
            loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))
            total_loss += loss.item()

            # Compute accuracy
            predictions = torch.argmax(logits, dim=-1)  # Get predicted classes
            correct_predictions += (predictions == targets).sum().item()
            total_predictions += targets.numel()

        # Compute average loss and accuracy
        avg_loss = total_loss / len(test_loader)
        accuracy = (correct_predictions / total_predictions) * 100

        print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")


/tmp/ipykernel_35320/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/tmp/ipykernel_35320/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)


Test Loss: 1.6178, Test Accuracy: 52.28%


Test Loss: 1.6178, Test Accuracy: 52.28%

## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [ ]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: Optional
        A scaling factor for randomness in predictions. Higher values (e.g., >1) make 
            predictions more random, while lower values (e.g., <1) make predictions more deterministic.
            Default is 1.0.
    """
    start_text = start_text.lower()
    #TODO: Implement the rest of the generate_text function
    model.eval()  # Set the model to evaluation mode
    generated_text = start_text
    input_text = start_text.lower()

    # Convert the starting text to indices
    input_indices = torch.tensor([char_to_idx[char] for char in input_text], dtype=torch.long).unsqueeze(0).to(device)

    # Initialize the hidden state
    hidden = model.init_hidden(batch_size=1)

    for _ in range(k):
        # Forward pass through the model
        logits, hidden = model(input_indices, hidden)

        # Take the last character's logits
        logits = logits[:, -1, :]  # [batch_size, vocab_size]

        # Sample a character from the output distribution
        sampled_idx = sample_from_output(logits, temperature)
        sampled_char = idx_to_char[sampled_idx.item()]

        # Append the character to the generated text
        generated_text += sampled_char

        # Prepare the next input
        input_indices = torch.tensor([[sampled_idx.item()]], dtype=torch.long).to(device)

    return generated_text

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.
Generated text: helloped up to be made to a quarterrow had stopped roun
Generated text: helloped to him to live the governski had sment with his hand the
Generated text: this man to see him and went and the same way and should be in the condition to returned to him in the s


ValueError: invalid literal for int() with base 10: ''

: 

## Report section

In your report, describe your experiments and observations when training the model with two datasets: (1) the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` and (2) the text from `warandpeace.txt`. Include the final loss values for both datasets and discuss how the generated text differed between the two. Explain the impact of changing the `temperature` parameter on the text generation, and provide examples. Reflect on the challenges you faced, your thought process during implementation, and the key insights you gained about RNNs and sequence modeling.


>Experiments and Observations
- In this project, we trained a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. The model was evaluated on two datasets to understand its ability to learn and generate text sequences: (1) a repetitive alphabet sequence ("abcdefghijklmnopqrstuvwxyz" * 100) and (2) text from "War and Peace." The repetitive alphabet dataset presented a simple and highly structured pattern, making it relatively easy for the model to learn. The final training loss on this dataset was remarkably low, at 0.0386, indicating that the model effectively captured the deterministic pattern of the data. During text generation, a low temperature (e.g., 0.5) produced highly accurate and repetitive sequences, such as "abcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyz". However, at a higher temperature (e.g., 1.5), the generated sequences became slightly jumbled but still recognizable, such as "abcdefgkwxyzlmnovhijqrstuvwxyzabcdetuvwx". This demonstrates how higher temperatures introduce randomness and creative variability at the expense of accuracy.

- In contrast, the second dataset, containing text from "War and Peace," posed a greater challenge due to the complexity and variability of natural language patterns. The model achieved a final loss of 1.5626, significantly higher than the alphabet dataset, reflecting the difficulty in capturing nuanced linguistic structures. At a low temperature (e.g., 0.5), the generated text exhibited coherent phrases and grammatically correct patterns, such as "the man walked into the room and sat down", though the content often lacked meaningful context. At a higher temperature (e.g., 1.5), the text was more diverse but less coherent, producing sentences like "he moon softly and cried world lighted hopes of night dreams". These results highlight the trade-offs between coherence and creativity in text generation, influenced by the temperature parameter.

> Impact of the Temperature Parameter
- The temperature parameter played a crucial role in controlling the randomness of the generated text. Lower temperatures led to more deterministic outputs closely resembling training patterns, making them suitable for tasks requiring accuracy and predictability. On the other hand, higher temperatures introduced greater randomness, resulting in creative but less structured outputs, which are better suited for exploratory or creative applications.

> Challenges, Thought Process, and Key Insights
- During implementation, several challenges were encountered. The model struggled with the variability and complexity of natural language in "War and Peace," highlighting the limitations of simple RNNs in capturing long-term dependencies. The vanishing gradient problem and limited memory of RNNs became apparent, particularly with longer and more complex sequences. Another challenge was finding the right hyperparameters to balance training efficiency and model performance. Despite these challenges, the experiments provided valuable insights. RNNs are effective at modeling short-term dependencies in structured data but face limitations in handling longer-term dependencies. Additionally, the importance of the temperature parameter became evident as a tool for balancing accuracy and creativity in text generation. Overall, this project provided a deeper understanding of sequence modeling and the trade-offs involved in text generation tasks.